### This is my first competition of this kind! The goal is to understand and learn!

## teachers dataset with teachers for the bot) Unfortunately, I couldn't figure out how to effectively set up games and play like a person from GitHub with 300,000 games. If anyone can provide guidance, I would appreciate it.

## orbit_liteV4 with additional features from github, just to see if it helps)

In [ ]:
# Шаг 1: Убить старую версию
!pip uninstall kaggle-environments -y

# Шаг 2: Установить новую БЕЗ кеша
!pip install --no-cache-dir "kaggle-environments>=1.28.0"

# Шаг 3: Проверить
import kaggle_environments
print(f"Версия: {kaggle_environments.__version__}")

In [ ]:
%%writefile main.py
from __future__ import annotations
import dataclasses
import os
import sys
from dataclasses import dataclass

# ============================================================
# ORBIT WARS - FUSION AGENT
# База: Producer V2 (reinforcement risk)
# Улучшения: Producer Hybrid v4 (4P бонусы + агрессия)
# Фишки GitHub: 5 итераций прицеливания + стартовая агрессия
# ============================================================


# Подключаем orbit_lite (путь к датасету)
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

import torch
from torch import Tensor

# Импорты из orbit_lite — проверенные функции для физики игры
from orbit_lite.geometry import fleet_speed
from orbit_lite.intercept_aim import intercept_angle
from orbit_lite.movement import MovementConfig, PlanetMovement
from orbit_lite.movement_step import (
    apply_private_planned_launches,
    concat_launch_entries,
    disambiguate_duplicate_launches,
    ensure_planet_movement,
    infer_planned_launches_from_entries,
)
from orbit_lite.obs import parse_obs
from orbit_lite.distance_cache import build_distance_cache
from orbit_lite.planner_core import (
    _candidate_indices,
    _empty_entries,
    _greedy_select,
    _plan_regroup,
    build_target_shortlist,
    capture_floor,
    empty_action_row,
    entries_to_sparse_payload,
    largest_initial_player_count,
    make_launch_set,
    reachable_mask,
    reinforcement_timing_factor,
    safe_drain,
    score_candidates,
)
from orbit_lite.adapter import single_obs_to_tensor, sparse_action_row_to_moves


# ============================================================
# КОНФИГУРАЦИЯ АГЕНТА
# ============================================================

@dataclass(frozen=True)
class ProducerLiteConfig:
    """
    Параметры стратегии.
    
    Ключевые настройки:
    - horizon: на сколько ходов вперёд моделируем движение
    - roi_threshold: минимальная окупаемость атаки
    - reinforce_size_beta: учёт вражеских подкреплений (β=2.2 из V2)
    - ffa_leader_attack_bonus: бонус атаки на лидера в 4P (из Hybrid v4)
    """
    
    # Горизонт планирования (как в V2 и Hybrid)
    horizon: int = 18
    
    # Размеры shortlist'ов
    max_sources_per_lane: int = 12
    max_offensive_targets: int = 12
    max_defensive_targets: int = 4
    
    # Параметры greedy-выбора волн
    max_waves_per_turn: int = 6
    roi_threshold: float = 1.5  # Порог окупаемости
    min_ships_to_launch: float = 4.0
    
    # ==========================================
    # ФИЧА ИЗ V2: Reinforcement risk
    # ==========================================
    # Не атакуем если враг может прислать подкрепление
    reinforce_size_beta: float = 2.2
    reinforce_eta_free: float = 3.0
    reinforce_eta_scale: float = 12.0
    
    # ==========================================
    # Regroup — переброска между своими планетами
    # ==========================================
    enable_regroup: bool = True
    max_regroup_time: float = 7.0
    regroup_pressure_delta_min: float = 0.25
    max_regroup_sources_per_lane: int = 6
    max_regroup_targets_per_source: int = 7
    regroup_pressure_norm: str = "none"
    regroup_time_penalty_weight: float = 1e-3
    
    # ==========================================
    # ФИЧА ИЗ HYBRID V4: 4P FFA бонусы
    # ==========================================
    ffa_leader_attack_bonus: float = 0.035
    ffa_target_prod_bonus: float = 0.08


# ============================================================
# КОНФИГУРАЦИЯ ДЛЯ 4 ИГРОКОВ (из Hybrid v4)
# ============================================================
CONFIG_4P = dataclasses.replace(
    ProducerLiteConfig(),
    horizon=13,                      # Меньше горизонт для 4P
    max_sources_per_lane=6,
    max_offensive_targets=7,         # Больше целей (из Hybrid)
    max_defensive_targets=2,
    roi_threshold=1.55,              # Агрессивнее (из Hybrid)
    min_ships_to_launch=5.0,        # Выше порог (из Hybrid)
    max_regroup_time=6.0,
    max_regroup_targets_per_source=8,
    ffa_leader_attack_bonus=0.035,  # Бонус атаки лидера (из Hybrid)
    ffa_target_prod_bonus=0.08,     # Бонус за производство цели (из Hybrid)
)


def _config_for(player_count: int) -> ProducerLiteConfig:
    """Выбираем конфиг в зависимости от числа игроков"""
    return CONFIG_4P if int(player_count) >= 4 else ProducerLiteConfig()


# ============================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def _movement_config(config: ProducerLiteConfig, *, player_count: int) -> MovementConfig:
    """Настройка моделирования движения планет"""
    return MovementConfig(
        movement_horizon=int(config.horizon),
        drift_epsilon=1e-3,
        track_fleets=True,
        player_count=int(player_count),
        max_tracked_fleets=128,
    )


def cheap_enemy_pressure(obs, cache, *, horizon: float, player_id: int) -> Tensor:
    """
    Оценка давления врага на каждую планету.
    
    Считает сколько вражеских кораблей может долететь до каждой
    планеты за horizon ходов. Используется для regroup и reinforcement.
    """
    P = int(obs.P)
    device = obs.device
    dtype = obs.ships.dtype
    if P == 0:
        return torch.zeros(P, dtype=dtype, device=device)
    
    d0 = cache.cross_dist[0].to(dtype)
    ships = obs.ships.to(dtype)
    speeds = fleet_speed(ships.clamp(min=1e-6))
    reach_dist = (speeds.view(P, 1) * float(horizon)).clamp(min=1e-6)
    
    enemy = obs.alive & (obs.owner_abs >= 0) & (obs.owner_abs != int(player_id))
    eye = torch.eye(P, device=device, dtype=torch.bool)
    valid = enemy.view(P, 1) & obs.alive.view(1, P) & ~eye
    
    decay = (1.0 - d0 / reach_dist).clamp(min=0.0)
    contrib = torch.where(valid, ships.view(P, 1) * decay, torch.zeros_like(decay))
    return contrib.sum(dim=0)


# ============================================================
# ОСНОВНОЙ ПЛАНИРОВЩИК ВОЛН
# ============================================================

def plan_lite_waves(
    *,
    movement: PlanetMovement,
    obs,
    obs_tensors: dict,
    cache,
    garrison_status,
    prod: Tensor,
    alive_by_step: Tensor,
    config: ProducerLiteConfig,
    player_count: int,
):
    """
    Планирует атаки и regroup для одного хода.
    
    1. Выбирает лучшие источники (планеты с кораблями)
    2. Выбирает лучшие цели (по расстоянию и ценности)
    3. Для каждой пары (источник, цель) считает:
       - Оптимальный угол перехвата (intercept_angle)
       - Сколько кораблей нужно для захвата (capture_floor)
       - Сколько можем безопасно отправить (safe_drain)
    4. Оценивает каждую атаку (score_candidates)
    5. Greedy-выбор лучших атак
    6. Regroup — переброска между своими планетами
    """
    P = obs.P
    device = obs.device
    dtype = obs.ships.dtype
    pid = int(obs.player_id)

    H_axis = int(garrison_status.ships.shape[-1])
    H = max(H_axis - 1, 0)
    K_eta = max(1, min(int(config.horizon), H))
    W = max(1, int(config.max_waves_per_turn))

    # Фильтруем источники: только наши живые планеты с достаточным гарнизоном
    source_mask = obs.owned & obs.alive & (obs.ships >= float(config.min_ships_to_launch))
    if not bool(source_mask.any()):
        return _empty_entries(device, dtype)

    S_cap = max(1, min(int(config.max_sources_per_lane), P))
    source_idx, source_exists = _candidate_indices(obs.ships, source_mask, S_cap)
    
    # Выбираем лучшие цели
    target_idx, target_exists = build_target_shortlist(
        obs, obs_tensors, garrison_status, cache,
        config=config, K_eta=K_eta, H=H, prod=prod, source_mask=source_mask,
    )
    if not bool(target_exists.any()):
        return _empty_entries(device, dtype)
    
    S = int(source_idx.shape[0])
    T = int(target_idx.shape[0])
    target_is_mine = obs.owned[target_idx.clamp(0, P - 1)]

    source_ships = obs.ships[source_idx.clamp(0, P - 1)].to(dtype)
    H_eff = torch.full((), float(H), dtype=dtype, device=device)
    
    # Безопасный слив: сколько можно отправить не оголяя планету
    drain = safe_drain(
        garrison_status, source_idx=source_idx,
        source_ships=source_ships, H_eff=H_eff, player_id=pid,
    )

    eta_cap = torch.full((T,), float(K_eta), dtype=dtype, device=device)

    # ==========================================
    # ФИЧА V2: Reinforcement risk
    # ==========================================
    beta = float(config.reinforce_size_beta)
    enemy_mass = (
        cheap_enemy_pressure(obs, cache, horizon=float(K_eta), player_id=pid)
        if beta > 0.0 or bool(config.enable_regroup) else None
    )

    reinforcement = None
    if beta > 0.0:
        enemy_mass_t = enemy_mass[target_idx.clamp(0, P - 1)]
        k_arange = torch.arange(1, K_eta + 1, device=device, dtype=dtype)
        rho = reinforcement_timing_factor(
            k_arange,
            eta_free=float(config.reinforce_eta_free),
            eta_scale=float(config.reinforce_eta_scale),
        )
        reinforcement = beta * rho.view(1, K_eta) * enemy_mass_t.view(T, 1)
    
    # Сколько кораблей нужно для захвата (с учётом reinforcement!)
    floor = capture_floor(
        garrison_status, target_idx=target_idx, k_max=K_eta,
        capture_overhead=1.0, player_id=pid,
        reinforcement=reinforcement,
    )
    K = int(floor.shape[-1])

    # Размер флота = безопасный слив с источника
    sizes = drain.view(S, 1).expand(S, T).floor()

    # Проверка досягаемости цели
    active = reachable_mask(
        movement, source_idx=source_idx, target_idx=target_idx,
        fleet_sizes=sizes.unsqueeze(-1), eta_cap=eta_cap,
    ).squeeze(-1)
    
    # ==========================================
    # ФИЧА GITHUB: 5 итераций прицеливания
    # (orbit_lite intercept_angle делает это внутри)
    # ==========================================
    aim = intercept_angle(
        movement,
        source_idx.unsqueeze(1),
        target_idx.unsqueeze(0),
        sizes,
        active=active,
    )
    angle = aim["angle"]
    eta = aim["eta"]
    viable = aim["viable"] & (eta <= eta_cap.view(1, T))

    # Проверка: хватит ли кораблей для захвата на момент прибытия
    if K > 0:
        k_arr = (eta.clamp(min=1.0, max=float(K)).ceil().long() - 1).clamp(0, K - 1)
        floor_at_arr = floor.unsqueeze(0).expand(S, T, K).gather(-1, k_arr.unsqueeze(-1)).squeeze(-1)
    else:
        floor_at_arr = torch.ones(S, T, dtype=dtype, device=device)
    clears_floor = sizes >= floor_at_arr

    src_neq_tgt = source_idx.view(S, 1) != target_idx.view(1, T)
    valid = (
        viable & clears_floor & (sizes >= 1.0) & src_neq_tgt
        & source_exists.view(S, 1) & target_exists.view(1, T)
    )

    # Формируем кандидатов для каждой пары (источник, цель)
    L = 1
    C = S * T
    cand_src = source_idx.view(S, 1).expand(S, T).reshape(C, L)
    cand_tgt_slot = target_idx.view(1, T).expand(S, T).reshape(C)
    cand_tgt_short = torch.arange(T, device=device).view(1, T).expand(S, T).reshape(C)
    cand_send = torch.where(valid, sizes, torch.zeros_like(sizes)).reshape(C, L)
    cand_angle = angle.reshape(C, L)
    cand_eta = torch.where(valid, eta, torch.ones_like(eta)).reshape(C, L)
    cand_active = valid.reshape(C, L)
    cand_valid = valid.reshape(C)
    cand_is_def = target_is_mine[cand_tgt_short]

    launches = make_launch_set(
        source_slots=cand_src,
        target_slots=cand_tgt_slot.unsqueeze(-1).expand(C, L),
        ships=cand_send,
        eta=cand_eta,
        valid=cand_active & cand_valid.unsqueeze(-1),
        player_id=pid,
    )
    
    # Оценка каждой атаки
    score = score_candidates(
        garrison_status, prod=prod, alive_by_step=alive_by_step,
        player_count=int(player_count), launches=launches, player_id=pid,
    )
    
    # ==========================================
    # ФИЧА ИЗ HYBRID V4: 4P FFA бонусы
    # ==========================================
    if int(player_count) >= 4 and (
        float(config.ffa_leader_attack_bonus) > 0.0
        or float(config.ffa_target_prod_bonus) > 0.0
    ):
        owner = obs.owner_abs.to(torch.long)
        owner_valid = (owner >= 0) & (owner < int(player_count)) & obs.alive
        owner_idx = owner.clamp(min=0, max=max(int(player_count) - 1, 0))
        prod_by_owner = torch.zeros(int(player_count), dtype=dtype, device=device)
        ships_by_owner = torch.zeros(int(player_count), dtype=dtype, device=device)
        prod_by_owner.scatter_add_(0, owner_idx, torch.where(owner_valid, prod.to(dtype), torch.zeros_like(prod.to(dtype))))
        ships_by_owner.scatter_add_(0, owner_idx, torch.where(owner_valid, obs.ships.to(dtype), torch.zeros_like(obs.ships.to(dtype))))
        strength = prod_by_owner + 0.025 * ships_by_owner
        my_strength = strength[pid].detach()

        target_owner = owner[target_idx.clamp(0, P - 1)].clamp(min=0, max=max(int(player_count) - 1, 0))
        target_owned_enemy = (
            target_exists
            & obs.is_enemy[target_idx.clamp(0, P - 1)]
            & (obs.owner_abs[target_idx.clamp(0, P - 1)] >= 0)
        )
        owner_strength = strength[target_owner]
        leader_delta = (owner_strength - my_strength).clamp(min=0.0)
        target_bonus_short = torch.where(
            target_owned_enemy,
            float(config.ffa_leader_attack_bonus) * leader_delta
            + float(config.ffa_target_prod_bonus) * prod[target_idx.clamp(0, P - 1)].to(dtype),
            torch.zeros_like(owner_strength),
        )
        score = score + target_bonus_short[cand_tgt_short]
    
    score = torch.where(cand_valid, score, torch.full_like(score, float("-inf")))

    # Greedy-выбор лучших атак
    wave_entries, leftover = _greedy_select(
        P=P, W=W, device=device, dtype=dtype, score=score,
        cand_src=cand_src, cand_send=cand_send, cand_angle=cand_angle,
        cand_eta=cand_eta, cand_active=cand_active,
        cand_tgt_slot=cand_tgt_slot, cand_tgt_short=cand_tgt_short,
        cand_is_def=cand_is_def, source_budget=obs.ships.to(dtype).clone(),
        target_exists=target_exists, roi_threshold=float(config.roi_threshold),
    )

    if not bool(config.enable_regroup):
        return wave_entries

    # Regroup: переброска оставшихся кораблей к плантациям под давлением
    regroup_entries = _plan_regroup(
        movement=movement, obs=obs, obs_tensors=obs_tensors,
        garrison_status=garrison_status, leftover=leftover,
        original_ships=obs.ships.to(dtype), pressure=enemy_mass,
        config=config, H=H,
    )
    return concat_launch_entries([wave_entries, regroup_entries])


# ============================================================
# ГЛАВНЫЙ ЦИКЛ ХОДА
# ============================================================

def run_turn(obs_tensors: dict, *, config: ProducerLiteConfig, player_count: int, memory) -> dict:
    """
    Обработка одного хода:
    1. Обновляем модель движения планет
    2. Планируем волны атак + regroup
    3. Применяем запуски к симуляции
    4. Возвращаем действия в формате для Kaggle
    """
    device = obs_tensors["planets"].device
    obs = parse_obs(obs_tensors)
    P = obs.P
    if P == 0:
        return empty_action_row(device)

    movement = ensure_planet_movement(
        obs_tensors=obs_tensors,
        expected_cfg=_movement_config(config, player_count=int(player_count)),
        cached_movement=getattr(memory, "movement", None),
    )
    memory.movement = movement
    
    cache = build_distance_cache(movement, max_k=int(config.horizon))
    H = int(config.horizon)
    status = movement.garrison_status(max_horizon=H)
    alive_by_step = movement.alive_by_step[: H + 1]

    entries = plan_lite_waves(
        movement=movement, obs=obs, obs_tensors=obs_tensors, cache=cache,
        garrison_status=status, prod=movement.planet_prod,
        alive_by_step=alive_by_step, config=config, player_count=int(player_count),
    )
    entries = disambiguate_duplicate_launches(entries)
    launches = infer_planned_launches_from_entries(
        obs_tensors=obs_tensors, movement=movement, entries=entries,
        player_id=int(obs.player_id),
    )
    apply_private_planned_launches(
        movement=movement, launches=launches,
        owner_id=int(obs.player_id), obs_tensors=obs_tensors,
    )
    planet_ids = obs_tensors["planets"][..., 0].long()
    return entries_to_sparse_payload(entries, planet_ids=planet_ids)


# ============================================================
# RUNTIME — КЕШ ДЛЯ УСКОРЕНИЯ
# ============================================================

class ProducerLiteMemory:
    """Хранит состояние между ходами для ускорения"""
    def __init__(self) -> None:
        self.movement = None
        self.cached_player_count: int | None = None
        self.last_sparse_action_row: dict | None = None

    def reset(self) -> None:
        self.movement = None
        self.cached_player_count = None
        self.last_sparse_action_row = None


class ProducerLiteRuntime:
    """Управляет памятью и вызовами"""
    def __init__(self, memory: ProducerLiteMemory | None = None) -> None:
        self.memory = memory if memory is not None else ProducerLiteMemory()

    def reset(self) -> None:
        self.memory.reset()

    def tensor_action(self, obs_tensors: dict):
        mem = self.memory
        if bool((obs_tensors["step"] == 0).all()):
            mem.cached_player_count = None
        if mem.cached_player_count is None:
            mem.cached_player_count = largest_initial_player_count(obs_tensors)
        config = _config_for(mem.cached_player_count)
        row = run_turn(
            obs_tensors, config=config,
            player_count=int(mem.cached_player_count), memory=mem,
        )
        mem.last_sparse_action_row = row
        return row


_RUNTIME = ProducerLiteRuntime()


# ============================================================
# ТОЧКА ВХОДА ДЛЯ KAGGLE
# ============================================================

def agent(obs):
    """
    Главная функция, вызываемая Kaggle каждый ход.
    
    Принимает наблюдение (obs) — словарь с планетами, флотами и т.д.
    Возвращает список действий: [[planet_id, angle, ships], ...]
    """
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    player_id = int(player)
    obs_tensors = single_obs_to_tensor(obs, player_id=player_id)
    with torch.no_grad():
        sparse_row = _RUNTIME.tensor_action(obs_tensors)
    return sparse_action_row_to_moves(sparse_row, obs, player_id=player_id)

In [ ]:
import shutil, tarfile, tempfile
from pathlib import Path

WORK = Path("/kaggle/working")
SRC = Path("/kaggle/input/datasets/evgendvorkin/orbit-lite/orbit_lite")

build = Path(tempfile.mkdtemp())
shutil.copy2(WORK / "main.py", build / "main.py")
shutil.copytree(SRC, build / "orbit_lite")

archive = WORK / "submission.tar.gz"
with tarfile.open(archive, "w:gz") as tar:
    tar.add(build / "main.py", arcname="main.py")
    tar.add(build / "orbit_lite", arcname="orbit_lite")

print(f"✅ Готово: {archive.stat().st_size:,} bytes")